In [ ]:
from BCP303.BCP303 import BPC303
from Sourcemeter.sourcemeter import Sourcemeter2401
import time
from datetime import datetime
from tool.tools import post_process


In [ ]:
# move the stage in Z direction
bcp303_z = BPC303(channel_id=2)
position_z = bcp303_z.move_to_origin(start_position=0)
step_size_z = 0.1  # move 0.1 mm
position_z = bcp303_z.bcp303_move_stage(step_size=step_size_z, current_position=position_z,)
bcp303_z.bcp303_stop(ifback=True)

In [ ]:
# move the stage in x direction
bcp303_x = BPC303(channel_id=1)
position_x = bcp303_x.move_to_origin(start_position=0)
step_size_x = 0.1  # move 0.1 mm
position_x = bcp303_x.bcp303_move_stage(step_size=step_size_x, current_position=position_x,)
bcp303_x.bcp303_stop(ifback=False)

In [ ]:
# record the signals
sm2401 = Sourcemeter2401(speed_nplc=0.1)


In [ ]:
# sweep operation: move stage in z-direction and record the signals

def sweep_operation(stage_settings=None,chip_name="chip_test",sample_name="beam_test"):
    step_size_x = stage_settings["step_size"]
    step_number = stage_settings["step_number"]
    time_interval = stage_settings["time_interval"]
    start_position = stage_settings["start_position"]
    step_size_z = stage_settings["step_size_z"]
    count = 1
    formatted_time = datetime.now().strftime("%Y%m%d%H%M")
    bcp303_z = BPC303(channel_id=2)
    bcp303_device = bcp303_z.get_device()
    # moving controller
    bcp303 = BPC303(channel_id=1, device=bcp303_device)
    # Sourcemeter
    sm2401 = Sourcemeter2401(speed_nplc=0.1)
    position_z = bcp303_z.move_to_origin(
        start_position=stage_settings["position_z"]
    )
    try:
        allData = [{"position": [], "voltage": []}]
        bcp303.move_to_origin()
        time.sleep(1)
        position_x = 0
        position_x = bcp303_x.bcp303_move_stage(step_size=step_size_x, current_position=position_x,)
        time.sleep(1)
        position = bcp303.move_to_origin(start_position=start_position)
        time.sleep(1)
        allData[0]["position"].append(position)
        voltage = sm2401.measure_voltage(duration=time_interval / 2, dt=0.01)[
            "voltage"
        ]
        allData[0]["voltage"].append(voltage)
        time.sleep(1)
        for step in range(step_number):
            step_start = time.perf_counter()
            position = bcp303.bcp303_move_stage(
                step_size=step_size_z,
                current_position=position,
            )
            allData[0]["position"].append(position)
            voltage = sm2401.measure_voltage(duration=time_interval / 2, dt=0.01)[
                "voltage"
            ]
            allData[0]["voltage"].append(voltage)
            # remaining time to wait until the next step
            elapsed = time.perf_counter() - step_start
            remaining = time_interval - elapsed
            if remaining > 0:
                time.sleep(remaining)
            print(f"process completed: {100 * count / step_number:.1f}%")
            count += 1
        sm2401_settings = sm2401.getSettings()
        stage_settings["position_z"] = position_z
        settings = {"stage": stage_settings, "sourcemeter": sm2401_settings}
        post_process(
            chip_name=chip_name,
            sample_name=sample_name,
            result=allData[0],
            config=settings,
            repeat=1,
            position_z=position_z,
            ifshow=False,
            formatted_time=formatted_time,
        )
    # stop the stage and sourcemeter
    except Exception as e:
        print(f"Exception occurred: {e}")
    finally:
        time.sleep(0.5)
        sm2401.close()
        bcp303_z.move_to_origin()
        bcp303_z.channel.StopPolling()
        bcp303.bcp303_stop(ifback=True)
        return allData[0], settings
    

setting = {
        "start_position": 2,
        "step_size": 0.005,
        "step_number": 400,
        "step_size_z": 1,
        "repeat_number": 5,
        "position_z": 0,
        "time_interval": 2,  
    }
sweep_operation(
        stage_settings=setting,
        chip_name="V1_R_W_2_Right",
        sample_name="w10_3_sweep",  
    )